In [1]:
#Load Data

# Import pandas, the main library for data analysis in Python
import pandas as pd

# Load the dataset into a dataframe called df
df = pd.read_csv("WA_Fn-UseC_-Telco-Customer-Churn.csv")

# Check the size of the dataset - rows and columns
print("Shape:", df.shape)

# Preview the first 5 rows to visually inspect the data
print("\nFirst 5 rows:")
df.head()

Shape: (7043, 21)

First 5 rows:


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [2]:
# Data Quality
 
# Check how many missing values exist in each column # A good dataset should have zeros across the board
print("Missing values per column:")
print(df.isnull().sum())

# Count how many customers churned vs stayed # This gives us our headline churn rate
print("\nChurn value counts:")
print(df['Churn'].value_counts())

Missing values per column:
customerID          0
gender              0
SeniorCitizen       0
Partner             0
Dependents          0
tenure              0
PhoneService        0
MultipleLines       0
InternetService     0
OnlineSecurity      0
OnlineBackup        0
DeviceProtection    0
TechSupport         0
StreamingTV         0
StreamingMovies     0
Contract            0
PaperlessBilling    0
PaymentMethod       0
MonthlyCharges      0
TotalCharges        0
Churn               0
dtype: int64

Churn value counts:
Churn
No     5174
Yes    1869
Name: count, dtype: int64


In [3]:
# Stastical Summary

# Get a statistical summary of all numeric columns # Shows min, max, average and spread of values
df.describe()

,SeniorCitizen,tenure,MonthlyCharges
count,7043.000000,7043.000000,7043.000000
mean,0.162147,32.371149,64.761692
std,0.368612,24.559481,30.090047
min,0.000000,0.000000,18.250000
25%,0.000000,9.000000,35.500000
50%,0.000000,29.000000,70.350000
75%,0.000000,55.000000,89.850000
max,1.000000,72.000000,118.750000


In [4]:
print(df['TotalCharges'].dtype)

str


In [5]:
# Convert TotalCharges to numeric, blank spaces become NaN
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')

# Check how many blank values that created
print("Missing TotalCharges after conversion:", df['TotalCharges'].isnull().sum())

# Fill those missing values with 0 (new customers with no charges yet)
df['TotalCharges'] = df['TotalCharges'].fillna(0)

print("Fixed! TotalCharges is now:", df['TotalCharges'].dtype)

Missing TotalCharges after conversion: 11
Fixed! TotalCharges is now: float64


In [6]:
# Convert Churn from Yes/No to 1/0 # 1 = churned, 0 = stayed
df['Churn'] = df['Churn'].map({'Yes': 1, 'No': 0})

print("Churn column converted:")
print(df['Churn'].value_counts())

Churn column converted:
Churn
0    5174
1    1869
Name: count, dtype: int64


In [7]:
# Group customers by contract type and calculate average churn rate
# Multiply by 100 to convert to a percentage

churn_by_contract = df.groupby('Contract')['Churn'].mean() * 100

print("Churn rate by contract type:")
print(churn_by_contract.round(2))

Churn rate by contract type:
Contract
Month-to-month    42.71
One year          11.27
Two year           2.83
Name: Churn, dtype: float64


In [8]:
# Calculate churn rate by payment method
# To understand if how customers pay affects their likelihood to leave
churn_by_payment = df.groupby('PaymentMethod')['Churn'].mean() * 100

print("Churn rate by payment method:")
print(churn_by_payment.round(2))

Churn rate by payment method:
PaymentMethod
Bank transfer (automatic)    16.71
Credit card (automatic)      15.24
Electronic check             45.29
Mailed check                 19.11
Name: Churn, dtype: float64


In [9]:
# Create tenure groups to segment customers by how long they've been with the company # This helps us see if newer customers churn more than older ones

df['tenure_group'] = pd.cut(df['tenure'], 
                            bins=[0, 12, 24, 48, 72], 
                            labels=['0-12 months', '13-24 months', '25-48 months', '49-72 months'])

# Calculate churn rate for each tenure group
churn_by_tenure = df.groupby('tenure_group')['Churn'].mean() * 100

print("Churn rate by tenure group:")
print(churn_by_tenure.round(2))

Churn rate by tenure group:
tenure_group
0-12 months     47.68
13-24 months    28.71
25-48 months    20.39
49-72 months     9.51
Name: Churn, dtype: float64


In [10]:
# Calculate churn rate by internet service type
churn_by_internet = df.groupby('InternetService')['Churn'].mean() * 100

print("Churn rate by internet service:")
print(churn_by_internet.round(2))

# Calculate average monthly charges for churned vs non-churned customers
churn_by_charges = df.groupby('Churn')['MonthlyCharges'].mean()

print("\nAverage monthly charges:")
print(churn_by_charges.round(2))

Churn rate by internet service:
InternetService
DSL            18.96
Fiber optic    41.89
No              7.40
Name: Churn, dtype: float64

Average monthly charges:
Churn
0    61.27
1    74.44
Name: MonthlyCharges, dtype: float64


In [11]:
# Average customer lifespan in months
avg_lifespan = df['tenure'].mean()

# Average monthly revenue per customer
avg_monthly_revenue = df['MonthlyCharges'].mean()

# Customer Lifetime Value = average monthly charge x average tenure
clv = avg_monthly_revenue * avg_lifespan

# Monthly Recurring Revenue at risk (churned customers x avg monthly charge)
churned_customers = df[df['Churn'] == 1]
mrr_at_risk = churned_customers['MonthlyCharges'].sum()

print(f"Average customer lifespan: {avg_lifespan:.1f} months")
print(f"Average monthly charge: ${avg_monthly_revenue:.2f}")
print(f"Customer Lifetime Value (CLV): ${clv:.2f}")
print(f"Monthly Revenue at risk (MRR): ${mrr_at_risk:,.2f}")

Average customer lifespan: 32.4 months
Average monthly charge: $64.76
Customer Lifetime Value (CLV): $2096.41
Monthly Revenue at risk (MRR): $139,130.85


In [12]:
# Revenue by internet service segment
revenue_by_segment = df.groupby('InternetService').agg(
    total_customers=('customerID', 'count'),
    avg_monthly_charge=('MonthlyCharges', 'mean'),
    total_monthly_revenue=('MonthlyCharges', 'sum'),
    churn_rate=('Churn', 'mean')
).round(2)

# Add monthly revenue at risk column
revenue_by_segment['monthly_revenue_at_risk'] = (
    revenue_by_segment['total_monthly_revenue'] * revenue_by_segment['churn_rate']
).round(2)

print("Revenue by internet service segment:")
print(revenue_by_segment)

Revenue by internet service segment:
                 total_customers  avg_monthly_charge  total_monthly_revenue  \
InternetService                                                               
DSL                         2421               58.10              140665.35   
Fiber optic                 3096               91.50              283284.40   
No                          1526               21.08               32166.85   

                 churn_rate  monthly_revenue_at_risk  
InternetService                                       
DSL                    0.19                 26726.42  
Fiber optic            0.42                118979.45  
No                     0.07                  2251.68  


In [13]:
# Create cohort based on tenure groups we already made # This shows what percentage of each group is still active
cohort_retention = df.groupby('tenure_group').agg(
    total_customers=('customerID', 'count'),
    churned=('Churn', 'sum'),
    retained=('Churn', lambda x: (x == 0).sum()),
    retention_rate=('Churn', lambda x: (x == 0).mean() * 100)
).round(2)

print("Cohort retention analysis:")
print(cohort_retention)

Cohort retention analysis:
              total_customers  churned  retained  retention_rate
tenure_group                                                    
0-12 months              2175     1037      1138           52.32
13-24 months             1024      294       730           71.29
25-48 months             1594      325      1269           79.61
49-72 months             2239      213      2026           90.49


In [14]:
# Export main cleaned dataset for Power BI
df.to_csv('customers_clean.csv', index=False)

# Export cohort retention table
cohort_retention.to_csv('cohort_retention.csv')

# Export revenue by segment table
revenue_by_segment.to_csv('revenue_by_segment.csv')

# Export churn by contract
churn_by_contract.to_csv('churn_by_contract.csv')

# Export churn by tenure group
churn_by_tenure.to_csv('churn_by_tenure.csv')

print("All files exported successfully!")
print("Files ready for Power BI:")
print("- customers_clean.csv")
print("- cohort_retention.csv")
print("- revenue_by_segment.csv")
print("- churn_by_contract.csv")
print("- churn_by_tenure.csv")

All files exported successfully!
Files ready for Power BI:
- customers_clean.csv
- cohort_retention.csv
- revenue_by_segment.csv
- churn_by_contract.csv
- churn_by_tenure.csv


In [15]:
# Create a proper cohort heatmap table # Extract the signup cohort month from tenure groups
df['cohort'] = pd.cut(df['tenure'], 
                      bins=[0, 12, 24, 48, 72],
                      labels=['0-12 months', '13-24 months', 
                              '25-48 months', '49-72 months'])

# Create cohort heatmap showing retention rate # by contract type and tenure group
cohort_heatmap = df.groupby(['Contract', 'cohort'])['Churn'].agg(
    retention_rate=lambda x: (x == 0).mean() * 100
).round(2).reset_index()

print(cohort_heatmap)
cohort_heatmap.to_csv('cohort_heatmap.csv', index=False)
print("\nExported successfully!")

          Contract        cohort  retention_rate
0   Month-to-month   0-12 months           48.65
1   Month-to-month  13-24 months           62.28
2   Month-to-month  25-48 months           67.08
3   Month-to-month  49-72 months           73.98
4         One year   0-12 months           89.43
5         One year  13-24 months           91.88
6         One year  25-48 months           89.38
7         One year  49-72 months           87.07
8         Two year   0-12 months          100.00
9         Two year  13-24 months          100.00
10        Two year  25-48 months           97.81
11        Two year  49-72 months           96.67

Exported successfully!
